# 1. - Подготовка

In [1]:

!git clone https://github.com/TebelevGt/Tool-Using-RL.git
%cd Tool-Using-RL/

Cloning into 'Tool-Using-RL'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 73 (delta 35), reused 54 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 22.79 KiB | 4.56 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/kaggle/working/Tool-Using-RL


In [2]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [3]:
#@title Colab Extra Install { display-mode: "form" }
#%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 55.3 MB/s eta 0:00:00:00:0100:01
Using Python 3.11.13 environment at: /usr
Resolved 31 packages in 64ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/553.12 KiB          
⠙ Preparing packages... (0/2)------------------- 14.90 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 30.90 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 46.90 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 62.90 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 75.40 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 91.40 KiB/553.12 KiB        
⠙ Preparing packages... (0/2)------------------- 107.40 KiB/553.12 KiB       
⠙ Preparing packages... (0/2)------------------- 123.40 KiB/553.12 KiB       
⠙ Preparing pac

In [4]:
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 214.8 MB/s eta 0:00:00
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.7.4
    Uninstalling unsloth_zoo-2025.7.4:
      Successfully uninstalled unsloth_zoo-2025.7.4
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.7.2
    Uninstalling unsloth-2025.7.2:
      Successfully uninstalled unsloth-2025.7.2


In [ ]:
import os
import json
#import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from envs import (
    PendolfEnv, 
PendolfDataset, 
grpo_env_reward_func,
get_pendolf_dataset
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-13 00:27:41.623312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773361661.822598      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773361661.878026      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 03-13 00:28:10 [__init__.py:244] Automatically detected platform cuda.
ERROR 03-13 00:28:14 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


# 2. GRPO Only model

In [7]:
def train_grpo_model(
    dataset_path: str,
    output_dir: str = "/kaggle/working/grpo_pendolf",
    num_generations: int = 4,
    model_name: str = "unsloth/Qwen2.5-1.5B-Instruct"
):
    """
    Обучает модель Qwen2.5-0.5B методом GRPO на заданном датасете.
    """

    # --- 1. Конфигурация модели ---
    max_seq_length = 4096 * 2
    lora_rank = 64

    print(f"--- Загрузка модели и токенизатора ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        fast_inference = True,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.8,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = lora_rank * 2,
        lora_dropout = 0.05,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = True,
    )

    tokenizer.padding_side = "right"

    # --- 2. Настройка GRPO ---
    print(f"--- Настройка GRPO Trainer (Dataset: {dataset_path}) ---")
    training_args = GRPOConfig(
        use_vllm = True,
        learning_rate = 5e-6,
        adam_beta1 = 0.9,
        adam_beta2 = 0.99,
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        num_generations = num_generations,
        max_prompt_length = 3000,
        max_completion_length = 1000,
        #max_steps = max_steps,
        #save_steps = max_steps,
        num_train_epochs = 1,
        save_strategy = "epoch",
        max_grad_norm = 0.1,
        report_to = "none",
        output_dir = output_dir,
        shuffle_dataset = False,
        temperature=0.8,     # <--- ДОБАВИТЬ! Заставит модель креативить
    )

    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            grpo_env_reward_func
        ],
        args = training_args,
        train_dataset = get_pendolf_dataset(dataset_path)
    )

    # --- 3. Обучение и сохранение ---
    print(f"--- Начало обучения ---")
    trainer.train()
    
    print(f"--- Сохранение модели в {output_dir} ---")
    trainer.save_model(output_dir)
    
    print(f"--- Сохранение истории обучения ---")
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/training_history.json", "w") as f:
        json.dump(trainer.state.log_history, f)
        
    return model, tokenizer

In [ ]:
model, tokenizer = train_grpo_model('data/curriculum/pendolf_train_curriculum.pkl')

--- Загрузка модели и токенизатора ---
INFO 03-13 00:28:29 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
INFO 03-13 00:28:29 [vllm_utils.py:753] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit with actual GPU utilization = 79.22%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 8192. Num Sequences = 48.
Unsloth: vLLM's KV Cache can use up to 10.05 GB. Also swap space = 4 G

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 03-13 00:28:50 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 03-13 00:28:50 [config.py:1472] Using max model len 8192
WARNING 03-13 00:28:50 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 03-13 00:28:51 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.0.self_attn', 'model.layers.1.mlp', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.7.mlp', 'model.layers.24.mlp', 'model.layers.26.mlp', 'model.layers.15.self_attn'], 'llm_int8_threshold': 6.0}
INFO 03-13 

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

--- Настройка GRPO Trainer (Dataset: data/curriculum/pendolf_train_curriculum.pkl) ---
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
--- Начало обучения ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 73,859,072 of 1,617,573,376 (4.57% trained)


INFO 03-13 00:29:56 [logger.py:59] Loading LoRA weights trained with rsLoRA.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / grpo_env_reward_func / mean,rewards / grpo_env_reward_func / std
1,-0.076100,0.431625,1.026813,35.250000,26.000000,47.000000,0.000000,35.250000,26.000000,47.000000,0.000000,0.431625,1.026813
2,-0.060900,0.816500,0.227709,33.000000,29.000000,35.000000,0.000000,33.000000,29.000000,35.000000,0.000000,0.816500,0.227709
3,-0.049700,0.402475,1.013577,37.250000,33.000000,44.000000,0.000000,37.250000,33.000000,44.000000,0.000018,0.402475,1.013577
4,0.577100,-1.117725,0.015712,63.750000,31.000000,136.000000,0.000000,63.750000,31.000000,136.000000,0.000024,-1.117725,0.015712
5,-0.216500,-0.098175,1.169792,54.000000,27.000000,101.000000,0.000000,54.000000,27.000000,101.000000,0.000106,-0.098175,1.169792
6,-0.141700,-0.096525,1.221560,34.750000,29.000000,45.000000,0.000000,34.750000,29.000000,45.000000,0.002518,-0.096525,1.221560
7,0.000500,-0.073950,1.193057,30.000000,21.000000,39.000000,0.000000,30.000000,21.000000,39.000000,0.002023,-0.073950,1.193057
8,0.978300,0.324100,1.285767,114.000000,31.000000,337.000000,0.000000,114.000000,31.000000,337.000000,0.005248,0.324100,1.285767
9,-0.075200,1.066400,0.204531,33.250000,28.000000,38.000000,0.000000,33.250000,28.000000,38.000000,0.005283,1.066400,0.204531
10,-0.074000,0.940925,0.046178,36.000000,24.000000,51.000000,0.000000,36.000000,24.000000,51.000000,0.007953,0.940925,0.046178


Exception ignored in: <function _xla_gc_callback at 0x7f9c6f477560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
